In [ ]:
## Very simpe implementation of langgraph done initially. not that much useful

In [1]:
from langgraph.graph import StateGraph
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
import os
import requests
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
class State(dict):
    pass

In [4]:
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectordb = Chroma(
    persist_directory="vector_db",
    embedding_function=embedding
)

C:\Users\KISHORE S\AppData\Local\Temp\ipykernel_16636\2005307146.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(
c:\Users\KISHORE S\anaconda3\envs\agent\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3205.23it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Note

In [13]:
def call_llm(prompt):
    response = requests.post(
        url="https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {os.getenv('OPENROUTER_API_KEY')}",
            "Content-Type": "application/json"
        },
        json={
            "model": "mistralai/mistral-7b-instruct",
            "messages": [
                {"role": "user", "content": prompt}
            ]
        }
    )
    
    #return response.json()["choices"][0]["message"]["content"]
    return response

In [14]:
call_llm("What is the capital of France?")

<Response [404]>

In [ ]:
def sponsor_node(state: State):
    
    user_input = state["input"]
    
    query = f"""
    Find sponsors for an {user_input['category']} conference 
    in {user_input['location']} with audience size {user_input['audience_size']}
    """
    
    # 🔍 Step 1: Retrieve from vector DB
    results = vectordb.similarity_search(query, k=10)
    
    context = "\n".join([r.page_content for r in results])
    
    # 🤖 Step 2: LLM reasoning
    prompt = f"""
    You are an expert event planner.
    
    Based on the following sponsor data:
    {context}
    
    Recommend the BEST 5 sponsors for this event:
    {user_input}
    
    Rank them and explain briefly.
    """
    
    answer = call_llm(prompt)
    
    state["sponsors"] = answer
    
    return state

In [ ]:
builder = StateGraph(State)

builder.add_node("sponsor", sponsor_node)

builder.set_entry_point("sponsor")

graph = builder.compile()

In [ ]:
if __name__ == "__main__":
    
    input_state = {
        "input": {
            "category": "AI",
            "location": "India",
            "audience_size": 2000
        }
    }
    
    result = graph.invoke(input_state)
    
    print("\n=== Recommended Sponsors ===\n")
    print(result["sponsors"])